# Computer Vision 101

เราจะสร้างระบบที่ดูภาพจากกล้องแล้วบอกว่า operator หยิบชิ้นงานไหนไปตรวจแล้วบ้าง ทำทีละขั้น:

1. หาแก้วในภาพ ตีกรอบรอบแก้ว
2. อ่านท่ามือ หาจุดบนมือ 21 จุด แล้วดูว่ากำหรือแบ
3. รวมกัน มือไปเกาะอยู่ที่แก้ว แปลว่ากำลังถือ
4. ให้เลขประจำตัวกับแก้วแต่ละใบ แล้วจำว่าใบไหนถูกหยิบไปตรวจแล้ว

รันทีละเซลล์จากบนลงล่าง ไม่ต้องแก้โค้ด

> อยากเปิด T4 GPU ก็ได้ (เมนู Runtime → Change runtime type) โน้ตบุ๊กจะเลือกใช้เอง ไม่เปิดก็รันได้ครบทุกเซลล์
> ตอนเปิดกล้องจะได้ราว ๆ 5 FPS พอกันทั้งสองแบบ ช้าเพราะอะไรเดี๋ยวเล่าให้ฟังตอนนั้น

In [ ]:
# ติดตั้งไลบรารี (ล็อกเวอร์ชันไว้ให้ผลเหมือนกันทุกเครื่อง)
# lapx = ตัวจับคู่ของ ByteTrack ที่พาร์ท 4 ใช้ — ลงตั้งแต่ตอนนี้ ไม่ให้ ultralytics ไปโหลดกลางคาบ
!pip install -q ultralytics==8.3.* mediapipe==1.0.1 lapx==0.5.*

In [ ]:
# เช็กว่าเครื่องมือครบและเวอร์ชันตรง
import sys, torch, ultralytics, mediapipe, cv2
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| GPU:", torch.cuda.is_available())
print("ultralytics:", ultralytics.__version__)
print("mediapipe  :", mediapipe.__version__)
assert ultralytics.__version__.startswith("8.3"), "ultralytics เวอร์ชันไม่ตรง"
print("\nพร้อมแล้ว เริ่มได้เลย —",
      "GPU" if torch.cuda.is_available() else "CPU", "(รันได้ครบทั้งสองแบบ)")

In [ ]:
# Colab เปิดกล้องตรง ๆ ไม่ได้ ต้องดึงภาพจากเบราว์เซอร์ผ่าน JavaScript
#   overlay=True  -> วิดีโอสดลื่น ~30fps + วาดผลเป็นเลเยอร์โปร่งใสทับ (ใช้ตอนทดสอบกล้อง)
#   overlay=False -> แสดงเฟรมที่ process_frame คืนมาตรง ๆ ตาม FPS จริงที่โมเดลรันได้
#                    (กล่องตรงกับเฟรมเป๊ะ ไม่ลอยตามหลัง — เห็นความเร็วจริงของ CPU)
import time
from base64 import b64decode, b64encode
import numpy as np, cv2, PIL.Image, io
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js

def _webcam_js():
    display(Javascript("""
      var video, div = null, stream, captureCanvas, layerImg, labelElement;
      var pendingResolve = null, shutdown = false;

      function removeDom() {
        stream.getVideoTracks()[0].stop();
        div.remove();
        video = null; div = null; stream = null;
        captureCanvas = null; layerImg = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            var ctx = captureCanvas.getContext('2d');
            ctx.save(); ctx.scale(-1, 1);
            ctx.drawImage(video, -640, 0, 640, 480);
            ctx.restore();
            result = captureCanvas.toDataURL('image/jpeg', 0.6);
          }
          var lp = pendingResolve; pendingResolve = null; lp(result);
        }
      }

      async function createDom() {
        if (div !== null) return stream;
        div = document.createElement('div');
        div.style.maxWidth = '640px';
        document.body.appendChild(div);

        labelElement = document.createElement('div');
        labelElement.style.fontWeight = 'bold';
        labelElement.innerText = 'กำลังเปิดกล้อง — ถ้าเบราว์เซอร์ถามสิทธิ์ ให้กด Allow';
        div.appendChild(labelElement);

        var stage = document.createElement('div');
        stage.style.position = 'relative';
        stage.style.width = '640px';
        div.appendChild(stage);

        video = document.createElement('video');
        video.width = 640;
        video.style.display = 'block';
        video.style.transform = 'scaleX(-1)';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };
        stream = await navigator.mediaDevices.getUserMedia({video: true});
        video.srcObject = stream;
        stage.appendChild(video);

        layerImg = document.createElement('img');
        layerImg.style.position = 'absolute';
        layerImg.style.top = '0px';
        layerImg.style.left = '0px';
        layerImg.style.width = '640px';
        layerImg.style.pointerEvents = 'none';
        stage.appendChild(layerImg);

        var hint = document.createElement('div');
        hint.innerHTML = '<span style="color:red;font-weight:bold;cursor:pointer">คลิกที่ภาพเพื่อหยุดกล้อง</span>';
        div.appendChild(hint);

        await video.play();

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640;
        captureCanvas.height = 480;
        window.requestAnimationFrame(onAnimationFrame);
        return stream;
      }

      async function stream_frame(label, data) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        await createDom();
        if (label !== "") labelElement.innerText = label;
        if (data !== "") layerImg.src = data;   // PNG โปร่งใส หรือ JPEG ทึบ แล้วแต่โหมด
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        return {'img': result};
      }
    """))

def _overlay_png(src, annotated):
    """เอาเฉพาะพิกเซลที่ process_frame วาดเพิ่ม มาทำเป็น PNG โปร่งใส"""
    mask = (cv2.absdiff(src, annotated).max(axis=2) > 8).astype(np.uint8) * 255
    bgra = cv2.cvtColor(annotated, cv2.COLOR_BGR2BGRA)
    bgra[:, :, 3] = mask
    _, buf = cv2.imencode(".png", bgra)
    return "data:image/png;base64," + b64encode(buf).decode()

def run_webcam(process_frame, seconds=20, overlay=True):
    """เปิดกล้อง เรียก process_frame(bgr) -> bgr ทุกเฟรม · คลิกที่ภาพ/ครบ seconds เพื่อหยุด
    overlay=True: วิดีโอสดลื่น วาดผลเป็นเลเยอร์โปร่งใส · overlay=False: โชว์เฟรมประมวลผลตรง ๆ FPS จริง"""
    _webcam_js()
    payload = ""
    n, t0 = 0, time.time()
    deadline = t0 + seconds
    try:
        while time.time() < deadline:
            fps = n / max(time.time() - t0, 1e-6)
            reply = eval_js('stream_frame("%.1f FPS", "%s")' % (fps, payload))
            if not reply or not reply.get("img"):
                break
            jpg = b64decode(reply["img"].split(",", 1)[1])
            bgr = cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)
            out = process_frame(bgr.copy())
            if overlay:
                payload = _overlay_png(bgr, out)
            else:
                _, buf = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, 60])
                payload = "data:image/jpeg;base64," + b64encode(buf).decode()
            n += 1
    except Exception as e:
        print("เปิดกล้องไม่ได้ —", repr(e))
        print("แก้: กด Allow ตอนเบราว์เซอร์ถาม / เปลี่ยนไปใช้ Chrome / รันเซลล์นี้ใหม่")
    finally:
        try:
            eval_js("shutdown = true")
            eval_js('stream_frame("", "")')
        except Exception:
            pass

def run_video(path, process_frame, seconds=20):
    """เล่นไฟล์วิดีโอแทนกล้อง (ใช้ตอนกล้องไม่ทำงาน)"""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    for _ in range(int(fps * seconds)):
        ok, frame = cap.read()
        if not ok:
            break
        _, buf = cv2.imencode(".jpg", process_frame(frame))
        clear_output(wait=True)
        display(PIL.Image.open(io.BytesIO(buf)))
    cap.release()

### การใช้กล้อง

- ตอนเบราว์เซอร์ถามหากล้อง กด **Allow**
- คลิกที่ภาพเพื่อหยุดกล้อง (หรือปล่อยให้ครบเวลาแล้วมันหยุดเอง)
- ใช้ Safari แล้วไม่ติด ลองเปลี่ยนไป Chrome
- ยังไม่ได้อีก อัดคลิปสั้น ๆ อัปโหลดเข้ามา แล้วเปลี่ยน `run_webcam(...)` เป็น `run_video("clip.mp4", ...)`

รันเซลล์ถัดไปเพื่อลองเปิดกล้อง ถ้าเห็นหน้าตัวเองพร้อมป้าย `camera OK` แปลว่าใช้ได้

In [ ]:
# ทดสอบกล้อง — รันตั้งแต่ต้นคาบ จะได้รู้ว่ากล้องใช้ได้ก่อนถึงพาร์ทที่ต้องใช้จริง
def camera_selftest(bgr):
    cv2.putText(bgr, "camera OK", (16, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
    return bgr

run_webcam(camera_selftest, seconds=8)

---
## 1. หาแก้วในภาพ

เราอยากรู้ว่าแก้วอยู่ตรงไหน คำตอบที่ได้คือกรอบสี่เหลี่ยมรอบแก้ว พร้อมค่าความมั่นใจ 0 ถึง 1

โมเดลที่จะใช้ (`yolo11n`) ผ่านการเทรนกับภาพนับแสนมาแล้ว รู้จักของอยู่ 80 อย่าง รวมถึงแก้วด้วย
เราไม่ได้สอนมันตั้งแต่ต้น แค่ปรับให้แม่นขึ้นกับแก้วและแสงในห้องนี้

In [ ]:
# โหลดรูปและ label
!git clone -q https://github.com/P-PrPas/tkk_workshop-data.git data
!ls data

In [ ]:
# label ต้องเป็น class 41 (cup ในชุด COCO) และพิกัดอยู่ในช่วง 0–1 — เช็กก่อนเทรน
from pathlib import Path
import matplotlib.pyplot as plt

splits = ["train", "val", "test"]
imgs = {s: sorted(Path(f"data/images/{s}").glob("*.jpg")) for s in splits}
image_names = {p.name for s in splits for p in imgs[s]}

for s in splits:
    for txt in Path(f"data/labels/{s}").glob("*.txt"):
        assert txt.with_suffix(".jpg").name in image_names, f"{txt.name} ไม่มีรูปคู่กัน"
        for line in txt.read_text().splitlines():
            if not line.strip():
                continue
            cls, *box = line.split()
            assert cls == "41", f"{txt.name}: class ต้องเป็น 41 (cup)"
            assert all(0 <= float(v) <= 1 for v in box), f"{txt.name}: พิกัดต้อง normalize 0-1"
print("label ผ่านการตรวจทั้งหมด")

# วาดกรอบจาก label ให้เห็นกับตา
all_imgs = [p for s in splits for p in imgs[s]]
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax, p in zip(axes.ravel(), all_imgs):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    lbl = Path(str(p).replace("/images/", "/labels/")).with_suffix(".txt")
    for line in lbl.read_text().splitlines() if lbl.exists() else []:
        if not line.strip():
            continue
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(im); ax.axis("off"); ax.set_title(p.name, fontsize=8)
for ax in axes.ravel()[len(all_imgs):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("train:", len(imgs["train"]), "| val:", len(imgs["val"]), "| test:", len(imgs["test"]))

### สอนด้วยรูปของเราเอง

รูปที่เตรียมไว้มีแก้วในห้องนี้ราว 10 ใบ กับรูปแก้วทั่วไปจากอินเทอร์เน็ตอีกไม่กี่สิบใบ
ฟังดูน้อย แต่ก็พอ เพราะโมเดลรู้จักแก้วอยู่แล้ว รูปพวกนี้แค่ช่วยจูนให้เข้ากับแก้วและแสงตรงหน้า

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")          # โมเดลนี้รู้จัก cup (คลาส 41) อยู่แล้ว
# amp=False: ปิด mixed-precision — บน GPU ค่า amp=True ทำให้ conf ที่ได้ต่ำผิดปกติ
model.train(data="data/cup.yaml", epochs=3, imgsz=640, batch=4, seed=0, amp=False, plots=True)

### ทำไมรูปแค่ไม่กี่สิบใบถึงพอ

ถ้าเราล้างความรู้เดิมของโมเดลทิ้ง แล้วสอนคำว่า "แก้ว" ใหม่จากศูนย์ด้วยรูปเท่านี้ มันจะแทบไม่เจออะไรเลย

ที่มันเวิร์ก เพราะเราเก็บความรู้เดิมไว้ทั้งหมด แล้วขยับแค่ส่วนที่เกี่ยวกับแก้ว
วิธีนี้เรียกว่า transfer learning งานจริงเกือบทั้งหมดก็ทำแบบนี้ แทบไม่มีใครเริ่มจากศูนย์

In [ ]:
# ผลบนรูปทดสอบที่โมเดลไม่เคยเห็นตอนเทรน
import matplotlib.pyplot as plt
test_imgs = sorted(Path("data/images/test").glob("*.jpg"))
fig, axes = plt.subplots(1, len(test_imgs), figsize=(6 * len(test_imgs), 6))
for ax, p in zip(np.atleast_1d(axes), test_imgs):
    r = model(str(p), conf=0.25, classes=[41], verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(p.name)
plt.tight_layout(); plt.show()

### แม่นแค่ไหน วัดยังไง

ดูรูปแล้วบอกว่า "ใช้ได้" มันไม่พอ เวลาส่งงานจริงเขาถามเป็นตัวเลข สองตัวที่ถามกันบ่อยที่สุดคือ

| | ถามว่า | สูตร | ต่ำแปลว่า |
|---|---|---|---|
| **Precision** | ที่ทายไปทั้งหมด ถูกกี่เปอร์เซ็นต์ | TP / (TP + FP) | ตีกรอบมั่ว เห็นอะไรก็ว่าแก้ว |
| **Recall** | ของจริงที่มีอยู่ เจอกี่เปอร์เซ็นต์ | TP / (TP + FN) | มองข้าม แก้ววางอยู่แต่ไม่เห็น |

- **TP** ทายว่ามีแก้ว แล้วมีจริง
- **FP** ทายว่ามีแก้ว แต่ไม่มี (ทายเกิน)
- **FN** มีแก้วอยู่ แต่ไม่ทาย (มองข้าม)

นับว่า "ตรงกัน" เมื่อกรอบที่ทายทับกรอบจริงเกินครึ่ง (IoU ≥ 0.5)

สองตัวนี้ขัดกันเสมอ ลด `conf` ลง Recall ขึ้น Precision ตก เพิ่ม `conf` ก็สลับกัน
งานคนละแบบเลือกคนละด้าน — ระบบตรวจงานยอมทายเกินดีกว่ามองข้าม (เอา Recall)
ระบบที่ทายผิดแล้วเสียหาย เอา Precision

In [ ]:
# นับ TP / FP / FN บนรูป test ด้วยมือ แล้วคำนวณ Precision กับ Recall ให้เห็นที่มา
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / max(union, 1e-9)

def truth_boxes(img_path, w, h):
    """อ่านกรอบจริงจากไฟล์ label แล้วแปลงกลับเป็นพิกเซล"""
    lbl = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    out = []
    for line in lbl.read_text().splitlines() if lbl.exists() else []:
        if line.strip():
            _, cx, cy, bw, bh = map(float, line.split())
            out.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
    return out

TP = FP = FN = 0
for p in test_imgs:
    h, w = cv2.imread(str(p)).shape[:2]
    truth = truth_boxes(p, w, h)
    matched = set()
    for pred in model(str(p), conf=0.25, classes=[41], verbose=False)[0].boxes.xyxy.tolist():
        best, idx = max(((iou(pred, t), i) for i, t in enumerate(truth) if i not in matched),
                        default=(0.0, -1))
        if best >= 0.5:
            TP += 1
            matched.add(idx)
        else:
            FP += 1                      # ทายกรอบนี้ไป แต่ไม่ตรงกับแก้วจริงใบไหนเลย
    FN += len(truth) - len(matched)      # แก้วจริงที่ไม่มีใครไปทับ = มองข้าม

precision = TP / max(TP + FP, 1)
recall = TP / max(TP + FN, 1)
print(f"TP (เจอจริง) {TP}   FP (ทายเกิน) {FP}   FN (มองข้าม) {FN}\n")
print(f"Precision = {TP}/({TP}+{FP}) = {precision:.2f}   ที่ทายไป ถูก {precision:.0%}")
print(f"Recall    = {TP}/({TP}+{FN}) = {recall:.2f}   แก้วจริงทั้งหมด เจอ {recall:.0%}")

fig, ax = plt.subplots(figsize=(6, 2.2))
ax.barh(["Recall", "Precision"], [recall, precision], color=["#f6a609", "#2e8b57"], height=0.55)
for i, v in enumerate([recall, precision]):
    ax.text(min(v + 0.02, 0.9), i, f"{v:.0%}", va="center", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1); ax.set_xticks([0, 0.5, 1]); ax.set_title("test set @ conf 0.25")
plt.tight_layout(); plt.show()

# ตัวเลขทางการจาก ultralytics (คิดทุก threshold ไม่ใช่แค่ 0.25) — ควรใกล้ ๆ กับที่นับเอง
metrics = model.val(split="test", classes=[41], verbose=False)
print("\nultralytics -> P", round(metrics.box.mp, 3),
      "| R", round(metrics.box.mr, 3),
      "| mAP50", round(metrics.box.map50, 3))

ตัวเลขพวกนี้มาจากรูปแค่ไม่กี่ใบ ผิดถูกรูปเดียวก็เด้งเป็นสิบเปอร์เซ็นต์ อย่าไปยึดกับมันมาก
ของจริงต้องใช้ test set หลักร้อยหลักพันรูปถึงจะพูดได้เต็มปาก

แต่ **วิธีนับ** ไม่ได้เปลี่ยนตามขนาดข้อมูล TP / FP / FN ก็คือ TP / FP / FN
พอเห็นว่ามันมาจากการนับกรอบทีละใบแบบนี้ เวลาเจอตารางผลจากที่อื่นก็อ่านออกแล้ว

### ลองกับกล้องจริง

เอาแก้วหลายแบบเข้าออกเฟรม เอียงดู เอามือบังบางส่วน

ลองเอาขวดน้ำมาวางข้าง ๆ ด้วย มันจะไม่ขึ้นกรอบ เพราะเราสั่งให้สนใจแค่แก้ว (`classes=[41]`)
ทั้งที่โมเดลก็รู้จักขวด การจงใจจำกัดขอบเขตแบบนี้ช่วยให้ระบบไม่เดามั่ว

ตั้งแต่นี้ภาพจะกระตุกเหลือราว 5 FPS (ดูเลขมุมบน) สลับไปใช้ T4 ก็ไม่ช่วย
เพราะที่ช้าไม่ใช่การรันโมเดล บน GPU ใช้เวลาแค่ไม่กี่มิลลิวินาที แต่เป็นการส่งภาพไปกลับ
ระหว่าง Python กับเบราว์เซอร์ผ่าน `eval_js` ของ Colab ตอนทดสอบกล้องเมื่อกี้ที่ลื่น
เพราะเป็นการดู `<video>` สดตรง ๆ ไม่ได้ส่งผ่านทางนี้ ส่วนแอปที่ต่อกล้องบนเครื่องเดียวกัน
ไม่ต้องส่งข้ามอะไร เลยได้ 30 FPS

In [ ]:
def process_frame(bgr):
    r = model(bgr, conf=0.25, classes=[41], verbose=False)[0]
    return r.plot()

run_webcam(process_frame, seconds=20, overlay=False)
# run_video("clip.mp4", process_frame)   # กล้องไม่ทำงาน? อัปโหลดคลิปแล้วใช้บรรทัดนี้แทน

---
## 2. อ่านท่ามือ

กรอบบอกได้แค่ว่ามีมืออยู่ตรงนี้ แต่ไม่บอกว่ามือทำท่าอะไร

รอบนี้เราจะหาจุดบนมือ 21 จุด แล้วดูจากตำแหน่งพวกนั้นว่ากำอยู่หรือแบอยู่

In [ ]:
# โหลดตัวตรวจจับมือ
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -la hand_landmarker.task

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

_opts = mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=mp_vision.RunningMode.VIDEO, num_hands=2)
landmarker = mp_vision.HandLandmarker.create_from_options(_opts)

# MediaPipe โหมดวิดีโอต้องการ timestamp (ms) ที่เพิ่มขึ้นเรื่อย ๆ ตลอด — ใช้นาฬิกาจริง
# จะได้ไม่ย้อนกลับตอนข้ามจากพาร์ท 2 ไปพาร์ท 3 (ใช้ landmarker ตัวเดียวกัน)
_last_ts = [0]
def video_ts():
    _last_ts[0] = max(_last_ts[0] + 1, int(time.monotonic() * 1000))
    return _last_ts[0]

TIPS = [4, 8, 12, 16, 20]     # ปลายนิ้วทั้งห้า
PIPS = [2, 6, 10, 14, 18]     # ข้อกลางของแต่ละนิ้ว

def count_extended(lm):
    """นับนิ้วที่เหยียด: ปลายนิ้วอยู่ไกลจากข้อมือกว่าข้อกลาง"""
    w = lm[0]
    d = lambda p: (p.x - w.x) ** 2 + (p.y - w.y) ** 2
    return sum(d(lm[t]) > d(lm[p]) for t, p in zip(TIPS, PIPS))

def hand_state(lm):
    n = count_extended(lm)
    return "FIST" if n <= 1 else "OPEN" if n >= 4 else "UNKNOWN"

### กำหรือแบ ดูจากระยะ

วิธีที่คนมักคิดก่อนคือเช็กว่าปลายนิ้วอยู่สูงกว่าข้อนิ้วไหม แต่พอเอียงมือหรือชี้ลงก็เพี้ยนแล้ว

เราวัดระยะจากข้อมือถึงปลายนิ้วแทน ถ้าปลายนิ้วอยู่ไกลกว่าข้อกลาง แปลว่านิ้วนั้นเหยียด หมุนมือยังไงก็ยังใช้ได้

In [ ]:
def hand_process_frame(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), video_ts())
    h, w = bgr.shape[:2]
    for lm in (res.hand_landmarks or []):
        pts = [(int(p.x * w), int(p.y * h)) for p in lm]
        for x, y in pts:
            cv2.circle(bgr, (x, y), 4, (0, 255, 0), -1)
        cv2.putText(bgr, hand_state(lm), pts[0], cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)
    return bgr

run_webcam(hand_process_frame, seconds=20, overlay=False)
# run_video("clip.mp4", hand_process_frame)

ลองกำแล้วแบช้า ๆ ดู ตอนมืออยู่กึ่งกลางระหว่างกำกับแบ ป้ายจะสั่นไปมา
อาการนี้จำไว้ก่อน เดี๋ยวส่วนที่ 3 เจออีก

---
## 3. รวมเป็น "กำลังถือแก้ว"

ไม่ต้องเทรนโมเดลใหม่ เอาผลจากสองส่วนก่อนหน้ามาต่อกัน

กฎแรกที่คิดไว้คือ "มือกำ และอยู่ตรงแก้ว" แต่พอลองจับแก้วที่ไม่มีหู มือแทบไม่ได้กำเลย
เลยเปลี่ยนมาดูว่าจุดบนมือไปตกอยู่ในกรอบแก้วหลายจุดพอไหม กำหรือแบไม่เกี่ยว

> จุดบนมืออยู่ในกรอบแก้วตั้งแต่ 8 จุดขึ้นไป แปลว่ากำลังถือ

In [ ]:
# กติกา "กำลังถือ": ดูว่าจุดบนมือไปตกอยู่ในกรอบแก้วกี่จุด
# เดิมเช็กด้วยว่ามือ "กำ" แต่จับแก้วที่ไม่มีหูมือแทบไม่กำ เลยตัดออก ดูแค่ว่ามือเกาะแก้วอยู่
def hand_on_cup(lm, w, h, cup_boxes, min_pts=8, margin=0.3):
    xs = [p.x * w for p in lm]
    ys = [p.y * h for p in lm]
    for x1, y1, x2, y2 in cup_boxes:
        mx, my = margin * (x2 - x1), margin * (y2 - y1)
        inside = sum(x1 - mx <= x <= x2 + mx and y1 - my <= y <= y2 + my
                     for x, y in zip(xs, ys))
        if inside >= min_pts:
            return True
    return False

In [ ]:
# รวมทั้งสองส่วนแล้วลองกับกล้อง
def combined_process_frame(bgr):
    h, w = bgr.shape[:2]
    r = model(bgr, conf=0.25, classes=[41], verbose=False)[0]
    cup_boxes = r.boxes.xyxy.tolist() if r.boxes is not None else []
    for x1, y1, x2, y2 in cup_boxes:
        cv2.rectangle(bgr, (int(x1), int(y1)), (int(x2), int(y2)), (255, 180, 0), 2)

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), video_ts())
    holding = False
    for lm in (res.hand_landmarks or []):
        on_cup = hand_on_cup(lm, w, h, cup_boxes)
        holding = holding or on_cup
        dot = (0, 255, 0) if on_cup else (0, 180, 255)   # เขียว = มือเกาะแก้ว
        for p in lm:
            cv2.circle(bgr, (int(p.x * w), int(p.y * h)), 3, dot, -1)

    label = "HOLDING" if holding else "NOT HOLDING"
    color = (0, 200, 0) if holding else (0, 0, 255)
    cv2.putText(bgr, label, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.6, color, 4)
    return bgr

run_webcam(combined_process_frame, seconds=20, overlay=False)
# run_video("clip.mp4", combined_process_frame)

---
## 4. โจทย์จริง: "ชิ้นไหนถูกหยิบไปตรวจแล้วบ้าง"

`HOLDING` / `NOT HOLDING` ตอบได้แค่ว่า *ตอนนี้* มีมือจับอะไรอยู่ไหม พอเอาไปวางในไลน์ผลิตจริง
คำถามของหัวหน้างานเป็นอีกแบบ:

> บนโต๊ะมีชิ้นงาน 5 ชิ้น operator หยิบไปตรวจครบหรือยัง ชิ้นไหนที่ยังไม่ได้แตะ

คำถามนี้ต้องการของที่เฟรมเดียวให้ไม่ได้ — ต้องรู้ว่า "แก้วใบนี้" กับ "แก้วใบเมื่อกี้" เป็นใบเดียวกัน
นั่นคือ **tracking**: ให้เลขประจำตัวกับของแต่ละชิ้น แล้วเลขนั้นต้องติดตัวมันไปทุกเฟรม

ของจริงมีอัลกอริทึมของมันอยู่แล้ว (เราใช้ ByteTrack) เปลี่ยนโค้ดคำเดียว `model(...)` → `model.track(...)`
แล้วผลลัพธ์จะมี `id` ติดมาให้เลย

สิ่งที่จะเห็นบนจอ

- **เลข id** บนแก้วแต่ละใบ
- **เส้นทาง** ที่แก้วใบนั้นเคลื่อนที่ผ่านมา
- **กรอบเหลือง** = ยังไม่ถูกหยิบ · **กรอบเขียว + ติ๊กถูก** = หยิบไปตรวจแล้ว (วางคืนก็ยังติ๊กอยู่)

In [ ]:
from collections import defaultdict, deque

trails = defaultdict(lambda: deque(maxlen=40))   # เส้นทางของแก้วแต่ละใบ (เก็บ 40 จุดล่าสุด)
picked = set()                                   # id ของแก้วที่ถูกหยิบไปตรวจแล้ว

GREEN, YELLOW = (60, 200, 60), (0, 200, 255)     # BGR: ตรวจแล้ว / ยังไม่ตรวจ

def draw_tick(bgr, x, y, col):
    """เครื่องหมายถูก — วาดเองสองเส้น ฟอนต์ของ OpenCV ไม่มีตัว ✓"""
    cv2.line(bgr, (x, y), (x + 7, y + 8), col, 3)
    cv2.line(bgr, (x + 7, y + 8), (x + 20, y - 10), col, 3)

def new_round():
    """เริ่มรอบตรวจใหม่ — ล้างเช็กลิสต์กับเส้นทางทิ้ง (ในแอปคือปุ่ม RESET บนหน้าจอ)"""
    trails.clear()
    picked.clear()

def inspection_frame(bgr):
    h, w = bgr.shape[:2]
    # track แทน predict: ผลลัพธ์มี .id ติดมาด้วย แก้วใบเดิมได้เลขเดิมทุกเฟรม
    r = model.track(bgr, persist=True, conf=0.25, classes=[41], verbose=False)[0]
    has_id = r.boxes is not None and r.boxes.id is not None
    ids = r.boxes.id.int().tolist() if has_id else []
    boxes = r.boxes.xyxy.tolist() if has_id else []

    res = landmarker.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)),
        video_ts())
    hands = res.hand_landmarks or []

    for tid, box in zip(ids, boxes):
        if any(hand_on_cup(lm, w, h, [box]) for lm in hands):
            picked.add(tid)              # หยิบแล้วคือหยิบแล้ว วางคืนก็ยังติ๊กถูกค้างไว้
        done = tid in picked
        col = GREEN if done else YELLOW
        x1, y1, x2, y2 = map(int, box)
        trails[tid].append(((x1 + x2) // 2, (y1 + y2) // 2))
        cv2.polylines(bgr, [np.array(trails[tid], np.int32)], False, col, 2)
        cv2.rectangle(bgr, (x1, y1), (x2, y2), col, 3)
        cv2.putText(bgr, f"cup {tid}", (x1, y1 - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)
        if done:
            draw_tick(bgr, x1 + 100, y1 - 20, col)

    for lm in hands:
        for p in lm:
            cv2.circle(bgr, (int(p.x * w), int(p.y * h)), 3, (255, 255, 255), -1)

    cv2.putText(bgr, f"checked {len(picked)}/{len(picked | set(ids))}", (16, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
    return bgr

new_round()                              # รันเซลล์นี้ซ้ำ = เริ่มรอบตรวจใหม่ทุกครั้ง
run_webcam(inspection_frame, seconds=30, overlay=False)
# run_video("clip.mp4", inspection_frame)

---
## ของจริงยากกว่านี้

ที่เราเพิ่งสร้างพอเห็นผลได้ แต่ยังห่างจากของที่ใช้งานจริงอีกไกล ลองสังเกตเอง:

- ป้ายสั่นตลอด เพราะระบบตัดสินใหม่ทุกเฟรม ไม่จำเฟรมก่อนหน้า
- เอามือเฉียดผ่านหน้าแก้วเฉย ๆ ก็ติ๊กถูกแล้ว ทั้งที่ยังไม่ได้หยิบขึ้นมาดูจริง ๆ
- มือกำบังแก้วจนตรวจไม่เจอ กล่องหายไปเลย พอกลับมามันได้ id ใหม่ กลายเป็นชิ้นใหม่ที่ยังไม่ตรวจ
- ช้า
- อยากเริ่มรอบตรวจใหม่ ต้องกดรันเซลล์ใหม่
- กล้องหลุดแล้วทุกอย่างค้าง

**การทำให้ "พอใช้ได้" นั้นง่าย แต่การทำให้ "ใช้งานได้จริง" นั้นยาก**
หกข้อนี้คือช่องว่างที่ว่า เดี๋ยวเราจะดูตัวที่แก้ครบทุกข้อ